# Influence maximization and propagation

In [ ]:
import json
import os
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "scripts/run_experiment.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find the project root")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.run_experiment import (
    DATA_DIR, DEFAULT_OUTPUT_DIR, ExperimentConfig, available_years,
    instance_mask, load_results, load_year_model, parameter_values, run_year, result_identity, validate_saved_seeds,
)
from src.gip_model import gip_from_seeds

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)


## Configuration

In [ ]:
YEARS = available_years(DATA_DIR)
if not YEARS:
    raise FileNotFoundError("Run python src/prepare_data.py first")
YEAR = max(YEARS)
CONFIG = replace(ExperimentConfig(), alpha=0.1, seed_budget=20)
RUN_SEARCH = True
VERBOSE_NADS = True
RESULTS_DIR = Path(os.environ.get("INFLUENCE_RESULTS_DIR", str(DEFAULT_OUTPUT_DIR))).expanduser().resolve()
RESULTS_PATH = RESULTS_DIR / "results.csv"

display(pd.Series({"year": YEAR, **parameter_values(CONFIG), **result_identity(YEAR, data_dir=DATA_DIR)}, name="value").to_frame())


## Yearly model

In [ ]:
model = load_year_model(YEAR, CONFIG, data_dir=DATA_DIR)
node_ids = model["node_ids"]
print(f"Year {YEAR}: {len(node_ids):,} municipalities, "
      f"{model['incidence'].shape[1]:,} companies, {model['incidence_count']:,} incidences")
print(f"Minimum company size: {CONFIG.min_edge_size}; "
      f"mean company weight: {model['edge_weights'].mean():.3f}; "
      f"mean municipality weight: {model['node_weights'].mean():.3f}")


In [ ]:
def simulate_ids(seed_ids):
    seed_ids = [int(value) for value in seed_ids]
    if len(seed_ids) != CONFIG.seed_budget or len(set(seed_ids)) != len(seed_ids):
        raise ValueError("Seed IDs must be unique and match CONFIG.seed_budget")
    missing = set(seed_ids) - set(node_ids)
    if missing:
        raise ValueError(f"Seed municipalities missing from this network: {sorted(missing)}")
    seeds = np.isin(node_ids, seed_ids).astype(float)
    return gip_from_seeds(
        model["incidence"], model["edge_weights"], seeds, CONFIG.gip_parameters,
        node_weights=model["node_weights"], stress_level=CONFIG.stress_level,
        alpha=CONFIG.alpha, max_iter=CONFIG.max_iter, budget=CONFIG.seed_budget,
    ).require_complete()


def propagation_table(diffusion):
    states = np.asarray(diffusion.states)
    active = states > 0
    reached = np.logical_or.accumulate(active, axis=0)
    weighted_state = states @ model["node_weights"]
    steps = np.arange(len(states))
    return pd.DataFrame({
        "step": steps,
        "active_now": active.sum(axis=1),
        "active_now_pct": 100 * active.mean(axis=1),
        "ever_reached": reached.sum(axis=1),
        "ever_reached_pct": 100 * reached.mean(axis=1),
        "population_weighted_state": weighted_state,
        "discounted_increment": (1 - CONFIG.gamma) ** steps * weighted_state,
        "cumulative_score": diffusion.spread_history,
    }).set_index("step")


## Optimization result

In [ ]:
if RUN_SEARCH:
    summary, _, _ = run_year(
        YEAR, CONFIG, verbose=VERBOSE_NADS, restart_search=False, data_dir=DATA_DIR,
    )
    start_ids, finish_ids = summary["start_list"], summary["finish_list"]
    start_score, finish_score = summary["initial_spread"], summary["best_spread"]
    search_history = summary["nads_history"]
    search_status = "time cap reached" if summary["search_time_limit_reached"] else "search completed before cap"
    print(f"Single NaDS search finished in {summary['search_elapsed_seconds']:.1f}s: {search_status}")
else:
    saved_results = load_results(RESULTS_PATH)
    matches = saved_results.loc[instance_mask(saved_results, {"year": YEAR, **parameter_values(CONFIG), **result_identity(YEAR, data_dir=DATA_DIR)})]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one saved result for this configuration at {RESULTS_PATH}; found {len(matches)}. "
            "Choose a saved CONFIG, run scripts/run_experiment.py, or set RUN_SEARCH=True."
        )
    saved_result = matches.iloc[0]
    validate_saved_seeds(saved_result, model, CONFIG)
    start_ids = json.loads(saved_result["start_list"])
    finish_ids = json.loads(saved_result["finish_list"])
    start_score, finish_score = saved_result["start_spread"], saved_result["finish_spread"]
    search_history = json.loads(saved_result["nads_history"])
    print(f"Loaded saved result from {RESULTS_PATH}")

selected_ids = [int(value) for value in finish_ids]
gain_pct = 100 * (finish_score / start_score - 1) if start_score else np.nan
display(pd.Series({
    "starting score": start_score, "optimized score": finish_score,
    "optimization gain (%)": gain_pct,
    "retained seeds": len(set(start_ids) & set(finish_ids)),
    "replaced seeds": len(set(start_ids) - set(finish_ids)),
}, name="value").to_frame())

history_rows = [
    {"search_number": search["search_number"], "random_seed": search["random_seed"], **event}
    for search in search_history
    for event in search["events"]
]
nads_events = pd.DataFrame(history_rows)
display(nads_events[["search_number", "call", "elapsed_seconds", "value", "seed_list"]])


## Propagation diagnostics

In [ ]:
start_diffusion = simulate_ids(start_ids)
best_diffusion = simulate_ids(finish_ids)
np.testing.assert_allclose(
    [start_diffusion.total_spread, best_diffusion.total_spread],
    [start_score, finish_score], rtol=1e-12, atol=1e-9,
    err_msg="Stored scores do not match the current data and propagation settings",
)
trajectories = {
    "Starting seeds": propagation_table(start_diffusion),
    "Optimized seeds": propagation_table(best_diffusion),
}
coverage_summary = pd.DataFrame([
    {
        "seed set": label,
        "score": diffusion.total_spread,
        "updates": diffusion.iterations,
        "stopping reason": diffusion.stopping_reason,
        "active at final step": int(table.iloc[-1]["active_now"]),
        "ever reached": int(table.iloc[-1]["ever_reached"]),
        "ever reached (%)": table.iloc[-1]["ever_reached_pct"],
        "complete reach": int(table.iloc[-1]["ever_reached"]) == len(node_ids),
    }
    for (label, table), diffusion in zip(trajectories.items(), [start_diffusion, best_diffusion])
]).set_index("seed set")
display(coverage_summary)
display(pd.concat(trajectories, names=["seed set", "step"]))


## Selected municipalities

In [ ]:
selected = model["node_audit"].reindex(selected_ids).reset_index()
selected.insert(0, "selection_rank", np.arange(1, len(selected) + 1))
selected["status"] = ["retained" if value in start_ids else "added" for value in selected_ids]
selected["structural_degree"] = selected["municipality_id"].map(model["structural_degree"])
display(selected[[
    "selection_rank", "municipality_id", "municipality_name", "population",
    "node_weight", "structural_degree", "status",
]])


## Propagation trajectories

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
for label, table in trajectories.items():
    axes[0].plot(table.index, table["active_now_pct"], marker="o", markersize=3, label=label)
    axes[1].plot(table.index, table["ever_reached_pct"], marker="o", markersize=3, label=label)
    axes[2].plot(table.index, table["cumulative_score"], marker="o", markersize=3, label=label)
axes[0].set(title="Municipalities active now", ylabel="Share of experiment network (%)", ylim=(0, 102))
axes[1].set(title="Municipalities reached at least once", ylabel="Share of experiment network (%)", ylim=(0, 102))
axes[2].set(title="Cumulative discounted influence", ylabel="Population-weighted score")
for axis in axes:
    axis.set_xlabel("Update (0 = seed state)")
    axis.legend()
fig.suptitle(f"Current propagation — {YEAR}, alpha={CONFIG.alpha:g}, k={CONFIG.seed_budget}")
plt.show()
